# Symmetric Parseval CNN — denoising experiments (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Theborna/symmetric_parseval_conv/blob/main/colab_experiments.ipynb)

This notebook trains the four core models (`baseline`, `symmetric`, `mirror`,
`symmetric_mirror`) at several noise levels and produces the paper's PSNR/SSIM
table (Markdown + LaTeX).

**Before you start:** enable a GPU via *Runtime → Change runtime type → GPU*,
and have your BSD500 `train.h5` / `test.h5` ready (see step 4 for ways to get
them onto Colab — Drive is only one option).


## 1. Check the runtime


In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4 is fine).')

Torch: 2.11.0+cu128 | device: cuda


## 2. Get the code


In [2]:
import os

REPO_DIR = 'symmetric_parseval_conv'
if not os.path.isdir(REPO_DIR) and os.path.basename(os.getcwd()) != REPO_DIR:
    !git clone https://github.com/Theborna/symmetric_parseval_conv.git
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)
!git pull --ff-only
print('Working dir:', os.getcwd())

Cloning into 'symmetric_parseval_conv'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 93 (delta 32), reused 80 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 832.49 KiB | 43.81 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Already up to date.
Working dir: /content/symmetric_parseval_conv


## 3. Install dependencies

Colab already ships `torch`, `torchvision` and `numpy`; we only add the lighter
packages the repo imports.


In [3]:
!pip install -q tqdm tensorboard h5py einops scikit-image matplotlib piqa pytorch-ssim

# utils/utilities.py imports pytorch_ssim at module load; make sure it resolves.
try:
    import pytorch_ssim, piqa  # noqa: F401
    print('SSIM deps OK')
except Exception as e:
    print('installing pytorch_ssim from source:', e)
    !pip install -q git+https://github.com/Po-Hsun-Su/pytorch-ssim.git

  Preparing metadata (setup.py) ... done
SSIM deps OK


## 4. Get your BSD500 data onto Colab

You need the pre-built HDF5 files `train.h5` and `test.h5` (the same ones you
train with locally). **Pick ONE option below** — each lands the files in
`/content/data/`, which the path cell at the end of this section expects. No
Google Drive account required.


### Option A — upload straight from your computer

Zero setup. Reliable for files up to a few hundred MB; slow/flaky for multi-GB
files (use Option B/C for those). A dialog will ask you to pick both files.


In [4]:
import os
os.makedirs('/content/data', exist_ok=True)
from google.colab import files
print('Select train.h5 and test.h5 ...')
uploaded = files.upload()
for fn in uploaded:
    os.replace(fn, f'/content/data/{fn}')
print('saved:', os.listdir('/content/data'))

Select train.h5 and test.h5 ...


KeyboardInterrupt: 

### Option B — download from a direct URL

Works with any direct link: Dropbox (append `?dl=1`), OneDrive, a personal /
university server, or a **GitHub Release asset** on your own repo (up to 2 GB
per file). Fill in the two URLs.


In [4]:
import os
os.makedirs('/content/data', exist_ok=True)
TRAIN_URL = 'https://github.com/Theborna/symmetric_parseval_conv/releases/download/data/train.h5'   # <-- direct link to train.h5
VAL_URL   = 'https://github.com/Theborna/symmetric_parseval_conv/releases/download/data/test.h5'   # <-- direct link to test.h5
assert TRAIN_URL and VAL_URL, 'set TRAIN_URL and VAL_URL first'
!wget -q --show-progress -O /content/data/train.h5 "{TRAIN_URL}"
!wget -q --show-progress -O /content/data/test.h5  "{VAL_URL}"
print('saved:', os.listdir('/content/data'))

/content/data/train 100%[===================>]   1.50G  7.90MB/s    in 3m 9s   
/content/data/test. 100%[===================>]  40.08M  7.04MB/s    in 9.3s    
saved: ['test.h5', 'train.h5']


### Option C — Hugging Face Hub (durable, good for reruns)

Upload the two files once to a (private) dataset repo, e.g. with
`huggingface_hub.HfApi().upload_file(...)`, then pull them here. Best if you'll
rerun this notebook often.


In [ ]:
!pip install -q huggingface_hub
import os, shutil
from huggingface_hub import hf_hub_download  # , login

# login('hf_xxx')          # uncomment for a PRIVATE dataset repo
HF_REPO = 'your-username/bsd500'   # <-- your dataset repo id

os.makedirs('/content/data', exist_ok=True)
for fn in ['train.h5', 'test.h5']:
    p = hf_hub_download(repo_id=HF_REPO, filename=fn, repo_type='dataset')
    shutil.copy(p, f'/content/data/{fn}')
print('saved:', os.listdir('/content/data'))

### Option D — Google Drive

Only if you do have Drive access on this account.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# then set TRAIN_H5/VAL_H5 below to your Drive paths

### Set the paths and patch the configs (required)

Run this after whichever option above you used. It rewrites the data paths in
`config.json` and every `experiment_configs/*.json`.


In [10]:
import glob, json

# Options A/B/C save here; for Drive (D) point these at your Drive paths.
TRAIN_H5 = '/content/data/train.h5'
VAL_H5   = '/content/data/test.h5'

BS = 480
num_workers = 2

assert os.path.exists(TRAIN_H5), f'train file not found: {TRAIN_H5}'
assert os.path.exists(VAL_H5),   f'val file not found: {VAL_H5}'

def patch_data_paths(train_h5, val_h5):
    for p in ['config.json'] + sorted(glob.glob('experiment_configs/*.json')):
        with open(p) as f:
            cfg = json.load(f)
        if 'training_options' in cfg:
            cfg['training_options']['train_data_file'] = train_h5
            cfg['training_options']['val_data_file'] = val_h5
            cfg['training_options']['batch_size'] = BS
            cfg['training_options']['num_workers'] = num_workers
            with open(p, 'w') as f:
                json.dump(cfg, f, indent=4)
            print('patched', p)

patch_data_paths(TRAIN_H5, VAL_H5)

patched config.json
patched experiment_configs/_template.json
patched experiment_configs/baseline.json
patched experiment_configs/mirror.json
patched experiment_configs/symmetric.json
patched experiment_configs/symmetric_mirror.json


## 5. Run the sweep

`EPOCHS` overrides every config (use `1` for a quick smoke test first). Leave
`CONFIGS` empty to run all four models, or set e.g. `--configs symmetric_mirror mirror`.
Each config's own `sigmas` (default `[5, 15, 25]`) are used.


In [11]:
EPOCHS  = 2              # bump up (e.g. 10) for the real run
OUTPUT  = 'exps/paper'
CONFIGS = ''             # e.g. '--configs symmetric_mirror mirror'

!python experiments.py -d {device} --epochs {EPOCHS} -o {OUTPUT} {CONFIGS}

2026-08-08 14:35:33.164455: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-08 14:35:33.236600: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

===== training baseline (baseline) @ sigma=5 =====
238400
Preparing the dataloaders
Building the model
BaselineParsevalCNN(
  (network): Sequential(
    (0): UnitaryMatrix(num_channels=1, out_channels=64, p=1)
    (1): LinearSpline(num_activations=64, init=identity, num_coeffs=51)
    (2): BCOP(64, 64, kernel_size=3, stride=1, bias=False, dilation=(1, 1))
    (3): Linear

## 6. Results


In [12]:
from IPython.display import Markdown, display
with open(os.path.join(OUTPUT, 'results.md')) as f:
    display(Markdown(f.read()))

# Denoising results (best validation metric)

| Model | sigma=5 (PSNR / SSIM) | sigma=15 (PSNR / SSIM) | sigma=25 (PSNR / SSIM) |
|---|---|---|---|
| Baseline (BCOP) | 36.68 / 0.9527 | 30.07 / 0.8384 | 27.56 / 0.7538 |
| Mirror | **36.85 / 0.9555** | 30.21 / 0.8447 | 27.60 / 0.7559 |
| Symmetric | 36.84 / 0.9556 | **30.26 / 0.8474** | 27.68 / 0.7612 |
| Symmetric Mirror | 36.84 / 0.9556 | 30.26 / 0.8476 | **27.69 / 0.7611** |


In [13]:
# Per-run view (best validation metrics + wall-clock minutes)
import pandas as pd
res = json.load(open(os.path.join(OUTPUT, 'results.json')))
rows = []
for name, d in res.items():
    for sigma, m in d.get('runs', {}).items():
        rows.append({'model': d.get('label', name), 'sigma': int(sigma),
                     'PSNR': round(m['best_psnr'], 2), 'SSIM': round(m['best_ssim'], 4),
                     'depth': m.get('depth'), 'width': m.get('width'),
                     'minutes': m.get('minutes')})
pd.DataFrame(rows).sort_values(['model', 'sigma']).reset_index(drop=True)

,model,sigma,PSNR,SSIM,depth,width,minutes
0,Baseline (BCOP),5,36.68,0.9527,16,64,62.83
1,Baseline (BCOP),15,30.07,0.8384,16,64,62.68
2,Baseline (BCOP),25,27.56,0.7538,16,64,62.56
3,Mirror,5,36.85,0.9555,8,64,62.65
4,Mirror,15,30.21,0.8447,8,64,62.56
5,Mirror,25,27.60,0.7559,8,64,62.61
6,Symmetric,5,36.84,0.9556,16,64,64.76
7,Symmetric,15,30.26,0.8474,16,64,64.56
8,Symmetric,25,27.68,0.7612,16,64,64.55
9,Symmetric Mirror,5,36.84,0.9556,8,64,63.43


In [14]:
# LaTeX table for the paper
print(open(os.path.join(OUTPUT, 'results.tex')).read())

\begin{table}[t]
\centering
\caption{Gaussian denoising on BSD500. PSNR (dB) / SSIM; higher is better. Best PSNR per noise level in bold.}
\label{tab:denoising}
\begin{tabular}{l cc cc cc}
\toprule
 & \multicolumn{2}{c}{$\sigma=5$} & \multicolumn{2}{c}{$\sigma=15$} & \multicolumn{2}{c}{$\sigma=25$} \\
\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}
Model & PSNR & SSIM & PSNR & SSIM & PSNR & SSIM \\
\midrule
Baseline (BCOP) & 36.68 & 0.9527 & 30.07 & 0.8384 & 27.56 & 0.7538 \\
Mirror & \textbf{36.85} & 0.9555 & 30.21 & 0.8447 & 27.60 & 0.7559 \\
Symmetric & 36.84 & 0.9556 & \textbf{30.26} & 0.8474 & 27.68 & 0.7612 \\
Symmetric Mirror & 36.84 & 0.9556 & 30.26 & 0.8476 & \textbf{27.69} & 0.7611 \\
\bottomrule
\end{tabular}
\end{table}



## 7. (Optional) Save the results

Download the outputs so they survive the Colab session.


In [15]:
from google.colab import files
files.download(os.path.join(OUTPUT, 'results.tex'))
files.download(os.path.join(OUTPUT, 'results.json'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>